# Toy Experiments

We want to perform ASR with Continuous Flow Matching. This is, predicting a sequence of categorical tokens by cross-attending to a continuous latent embedding that contains information about the whole sequence.

To do this, we want to consider the following toy experiment. Let's create a vocabulary of $|\mathcal{V}|$ tokens and try to predict sequences of L tokens.

Let's say valid sequences are the ones in which the $\sum_\ell w^\ell = 0$.

We want to condition on $a^\ell = e_{w^\ell} + \eta \, \varepsilon^\ell, \varepsilon \sim \mathcal{N}(0, I)$ with $\eta$ controlling the noise level.

In [ ]:
import math
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")


### Define HPs

In [2]:
VOCAB_SIZE = 15
SEQ_LENGTH = 8
NUM_SAMPLES = 10000
NUM_VALID_TRANSITIONS = 4
REQUIRE_EVEN_SUM = True

### Create Dataset

In [3]:
def create_transition_matrix(vocab_size, num_valid_transitions, device):
    # row indices: each row repeated k times -> shape (V, k)
    row_idx = torch.arange(vocab_size, device=device).unsqueeze(1).expand(-1, num_valid_transitions)

    # distinct successors per row, sampled without replacement
    col_idx = torch.stack([
        torch.randperm(vocab_size)[:num_valid_transitions]
        for _ in range(vocab_size)
    ])

    transition_matrix = torch.zeros((vocab_size, vocab_size), dtype=torch.int32, device=device)
    transition_matrix[row_idx, col_idx] = 1

    # each row has exactly k successors
    assert (transition_matrix.sum(dim=1) == num_valid_transitions).all()

    return transition_matrix

def count_valid(trans_matrix, seq_length):
    ones = torch.ones(trans_matrix.shape[0], device=trans_matrix.device)
    M = torch.linalg.matrix_power(trans_matrix, seq_length - 1)
    return int((ones @ M @ ones).item())


def sample_sequence(
    trans_matrix,
    sample_size,
    seq_length,
    device=None,
    require_even_sum=True
):
    device = device or trans_matrix.device
    vocab_size = trans_matrix.shape[0]

    # Uniform distribution
    first_token_dist = torch.ones(vocab_size, device=device) / vocab_size

    # Sample first token
    w = torch.zeros(sample_size, seq_length, dtype=torch.long, device=device)
    w[:, 0] = torch.multinomial(first_token_dist, sample_size, replacement=True)

    # Fill the rest of the sequence
    for l in range(1, seq_length):
        probs = trans_matrix[w[:, l - 1]]
        w[:, l] = torch.multinomial(probs, 1).squeeze(1)

    if require_even_sum:
        even_sum_mask = (w.sum(dim=1) % 2 == 0)
        w = w[even_sum_mask]

    return w


In [5]:
transition_matrix = create_transition_matrix(VOCAB_SIZE, NUM_VALID_TRANSITIONS, device=device)
valid_seq_count = count_valid(transition_matrix.float(), seq_length=SEQ_LENGTH)

print(
    f"Valid sequences of length {SEQ_LENGTH}: {valid_seq_count}"
)

# row-normalize for sampling
prob_trans_matrix = transition_matrix.float() / transition_matrix.sum(dim=1, keepdim=True)

all_sequences = torch.empty(0, SEQ_LENGTH, dtype=torch.long, device=device)

if REQUIRE_EVEN_SUM:
    print("Sampling only sequences with even sum.")
    valid_seq_count = valid_seq_count // 2  # Adjust this as needed

while all_sequences.shape[0] < valid_seq_count:
    new_samples = sample_sequence(
        prob_trans_matrix, NUM_SAMPLES, SEQ_LENGTH, device=device, require_even_sum=REQUIRE_EVEN_SUM
    )
    all_sequences = torch.cat([all_sequences, new_samples], dim=0)
    all_sequences = torch.unique(all_sequences, dim=0)
    print(f"Sampled {all_sequences.shape[0]} unique sequences of length {SEQ_LENGTH} out of {valid_seq_count}.")


Valid sequences of length 8: 245760
Sampling only sequences with even sum.
Sampled 4805 unique sequences of length 8 out of 122880.
Sampled 9493 unique sequences of length 8 out of 122880.
Sampled 13987 unique sequences of length 8 out of 122880.
Sampled 18473 unique sequences of length 8 out of 122880.
Sampled 22691 unique sequences of length 8 out of 122880.
Sampled 26666 unique sequences of length 8 out of 122880.
Sampled 30452 unique sequences of length 8 out of 122880.
Sampled 34091 unique sequences of length 8 out of 122880.
Sampled 37639 unique sequences of length 8 out of 122880.
Sampled 41083 unique sequences of length 8 out of 122880.
Sampled 44300 unique sequences of length 8 out of 122880.
Sampled 47474 unique sequences of length 8 out of 122880.
Sampled 50459 unique sequences of length 8 out of 122880.
Sampled 53343 unique sequences of length 8 out of 122880.
Sampled 56097 unique sequences of length 8 out of 122880.
Sampled 58707 unique sequences of length 8 out of 122880.

KeyboardInterrupt: 

### Visualize how corruption messes w predictions

In [ ]:
def check_valid(sequences, transition_matrix, require_even_sum=True):
    # Cut first and last tokens to get pairs of transitions
    prev = sequences[:, :-1]
    nxt  = sequences[:, 1:]

    # entry [b, l] = 1 iff (prev[b,l] -> nxt[b,l]) is a legal transition
    legal = transition_matrix[prev, nxt].bool()

    per_sequence_valid = legal.all(dim=1)   # (B,) — every transition must hold

    if require_even_sum:
        parity_valid = (sequences.sum(dim=1) % 2 == 0)
        per_sequence_valid = per_sequence_valid & parity_valid

    num_valid = per_sequence_valid.sum().item()
    frac_valid = per_sequence_valid.float().mean().item()

    return num_valid, frac_valid, per_sequence_valid


def linear_interpolant_simulation(sequences, vocab_size, t):
    onehot = F.one_hot(sequences, num_classes=vocab_size).float()
    noisy = t * onehot + (1 - t) * torch.randn_like(onehot)
    return noisy.argmax(dim=-1)

import matplotlib.pyplot as plt

t_values = torch.linspace(0.0, 1.0, 21)
frac_valid_list = []

for t in t_values:
    noisy_samples = linear_interpolant_simulation(
        all_sequences, vocab_size=transition_matrix.shape[0], t=t.item()
    )
    _, frac_valid, _ = check_valid(noisy_samples, transition_matrix, require_even_sum=REQUIRE_EVEN_SUM)
    frac_valid_list.append(frac_valid)

plt.figure(figsize=(6, 4))
plt.plot(t_values.numpy(), frac_valid_list, marker="o")
plt.xlabel("t")
plt.ylabel("Fraction of valid sequences")
plt.title("Validity of argmax(I_t) as a function of t")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Number of valid samples: 66712 out of 122893 (54.28%)


In [104]:

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):  # t: (B,)
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t[:, None] * freqs[None, :]  # (B, half)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, dim)


class DenoiserBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # bidirectional self-attention — no causal mask, the model sees all positions
        h = self.norm1(x)
        x = x + self.attn(h, h, h, need_weights=False)[0]
        x = x + self.ff(self.norm2(x))
        return x

class Denoiser(nn.Module):
    def __init__(self, vocab_size, seq_len, audio_dim=None, d_model=128, n_heads=4, n_layers=3, d_ff=256):
        super().__init__()
        audio_dim = audio_dim if audio_dim is not None else vocab_size  # default: plain one-hot proxy
        self.x_proj = nn.Linear(vocab_size, d_model)
        self.a_proj = nn.Linear(audio_dim, d_model)
        self.time_embed = SinusoidalTimeEmbedding(d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        self.blocks = nn.ModuleList([DenoiserBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x_t, t, a):
        # x_t: (B, L, V), t: (B,), a: (B, L, audio_dim)
        h = self.x_proj(x_t) + self.a_proj(a) + self.pos_embed
        h = h + self.time_embed(t)[:, None, :]
        for block in self.blocks:
            h = block(h)
        return self.head(self.norm_out(h))

class AudioDecoder(nn.Module):
    """r_psi: predicts per-token categorical distributions from audio alone.
    Strictly factorized across positions — no attention, no dependence on x_t or t.
    """
    def __init__(self, audio_dim, vocab_size, hidden_dim=128, n_layers=2):
        super().__init__()
        layers = []
        in_dim = audio_dim
        for _ in range(n_layers - 1):
            layers += [nn.Linear(in_dim, hidden_dim), nn.GELU()]
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, vocab_size))
        self.net = nn.Sequential(*layers)

    def forward(self, a):
        return self.net(a)

In [108]:
def make_epoch_loader(sequences, batch_size, shuffle):
    return torch.utils.data.DataLoader(sequences, batch_size=batch_size, shuffle=shuffle)


def make_interpolant(x1, t, alpha_fn, beta_fn, gamma_fn=None, mu=None, device=None):
    device = device if device is not None else x1.device
    x0 = torch.randn_like(x1, device=device)
    I_t = alpha_fn(t)[:, None, None] * x0 + beta_fn(t)[:, None, None] * x1
    if gamma_fn is not None and mu is not None:
        I_t = I_t + gamma_fn(t)[:, None, None] * mu
    return I_t


def run_epoch(
    model,
    loader,
    transition_matrix,
    device,
    interpolant_fn,
    optimizer=None,
    scheduler=None,
    sigma=0.5,
    audio_proxy_fn=None,
    require_even_sum=True,
    mu_nn=None,
    mu_loss_weight=1.0,
):
    """One pass over `loader`. If optimizer is given, trains; otherwise evaluates.
    Returns a dict of epoch-averaged metrics."""
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss, total_seqs = 0.0, 0
    total_valid, total_unique_valid = 0, 0

    ctx = torch.enable_grad() if train_mode else torch.no_grad()
    with ctx:
        for batch in loader:
            batch = batch.to(device)
            B = batch.size(0)
            vocab_size = model.head.out_features

            x1 = F.one_hot(batch, vocab_size).float()

            if audio_proxy_fn is not None:
                a = audio_proxy_fn(batch)
            else:
                a = x1 + sigma * torch.randn_like(x1)

            if mu_nn is not None:
                mu = mu_nn(a)
            else:
                mu = None

            t = torch.rand(B, device=device)
            I_t = interpolant_fn(x1=x1, t=t, mu=mu, device=device)

            logits = model(I_t, t, a)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), batch.reshape(-1))

            if mu_nn is not None:
                mu_loss = F.cross_entropy(mu.reshape(-1, vocab_size), batch.reshape(-1))
                loss += mu_loss_weight * mu_loss

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()

            pred_sequences = logits.argmax(dim=-1)
            num_valid, _, valid_mask = check_valid(pred_sequences, transition_matrix, require_even_sum=require_even_sum)
            num_unique_valid = torch.unique(pred_sequences[valid_mask], dim=0).size(0) if num_valid > 0 else 0

            total_loss += loss.item() * B
            total_seqs += B
            total_valid += num_valid
            total_unique_valid += num_unique_valid

    return {
        "loss": total_loss / total_seqs,
        "frac_valid": total_valid / total_seqs,
        "unique_valid_count": total_unique_valid,   # not a fraction across batches — see note below
    }

In [85]:
def make_phi(
    vocab_size,
    embed_dim,
    num_confusable_pairs=1,
    confusion_noise=0.1,
    device=None,
    generator=None
):
    assert 2 * num_confusable_pairs <= vocab_size, (
        f"Can't form {num_confusable_pairs} disjoint"
        f"pairs from a vocab of size {vocab_size}"
    )

    Phi = torch.randn(
        vocab_size, embed_dim, device=device, generator=generator
    )

    # sample num_confusable_pairs disjoint pairs from V tokens
    perm = torch.randperm(
        vocab_size, generator=generator
    )[: 2 * num_confusable_pairs]
    pairs = perm.view(num_confusable_pairs, 2)  # (num_pairs, 2), each row a disjoint pair

    for i, j in pairs:
        Phi[j] = Phi[i] + confusion_noise * torch.randn(embed_dim, device=device, generator=generator)

    return Phi, pairs

def make_audio_proxy(w, Phi, kernel, eta, audio_dim):
    emb = Phi[w] # (B, L, d)
    emb_t = emb.transpose(1, 2) # (B, d, L)
    smoothed = F.conv1d(
        emb_t,
        kernel.expand(audio_dim, 1, -1),
        padding=kernel.shape[-1]//2,
        groups=audio_dim
    )
    smoothed = smoothed.transpose(1, 2) # (B, L, d)
    return smoothed + eta * torch.randn_like(smoothed, device=smoothed.device)


In [ ]:
num_epochs = 50
bs = 2048
sigma = 0.5
max_lr = 1e-2
audio_dim = 10

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

perm = torch.randperm(all_sequences.size(0))
all_sequences = all_sequences[perm]

train_size = int(0.8 * all_sequences.size(0))
train_data = all_sequences[:train_size]
val_data = all_sequences[train_size:]

train_loader = make_epoch_loader(train_data, bs, shuffle=True)
val_loader = make_epoch_loader(val_data, bs, shuffle=False)

model = Denoiser(vocab_size=vocab_size, seq_len=SEQ_LENGTH, audio_dim=audio_dim).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr)

interpolant_fn = partial(
    make_interpolant,
    alpha_fn=lambda t: 1 - t,
    beta_fn=lambda t: t,
)

steps_per_epoch = len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=max_lr,
    total_steps=steps_per_epoch * num_epochs,
)

history = {"train": [], "val": []}

confusion_noise = 0.3
Phi, confusable_pairs = make_phi(
    vocab_size, audio_dim, num_confusable_pairs=2, confusion_noise=confusion_noise, device=device
)
print(confusable_pairs)

eta = 1
kernel = torch.tensor([0.1, 0.8, 0.1], device=device).view(1, 1, 3)
audio_proxy_fn = partial(
    make_audio_proxy, Phi=Phi, kernel=kernel, eta=eta, audio_dim=audio_dim
)

for epoch in range(num_epochs):
    train_metrics = run_epoch(
        model,
        train_loader,
        transition_matrix,
        device,
        optimizer=optimizer,
        scheduler=scheduler,
        sigma=sigma,
        audio_proxy_fn=audio_proxy_fn,
        require_even_sum=require_even_sum,
        interpolant_fn=interpolant_fn
    )
    val_metrics = run_epoch(
        model,
        val_loader,
        transition_matrix,
        device,
        optimizer=None,
        scheduler=None,
        sigma=sigma,
        audio_proxy_fn=audio_proxy_fn,
        require_even_sum=require_even_sum,
        interpolant_fn=interpolant_fn
    )

    history["train"].append(train_metrics)
    history["val"].append(val_metrics)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"train loss {train_metrics['loss']:.4f} valid% {train_metrics['frac_valid']*100:.1f} | "
        f"val loss {val_metrics['loss']:.4f} valid% {val_metrics['frac_valid']*100:.1f}"
    )

tensor([[ 7, 14],
        [ 4, 10]])
Epoch 1/50 | train loss 1.3189 valid% 7.3 | val loss 0.8254 valid% 13.4
Epoch 2/50 | train loss 0.7569 valid% 17.4 | val loss 0.7152 valid% 20.4
Epoch 3/50 | train loss 0.6916 valid% 22.5 | val loss 0.6376 valid% 27.3
Epoch 4/50 | train loss 0.5767 valid% 33.4 | val loss 0.5401 valid% 36.9
Epoch 5/50 | train loss 0.5208 valid% 38.0 | val loss 0.6948 valid% 31.1
Epoch 6/50 | train loss 0.5495 valid% 35.8 | val loss 0.5786 valid% 36.7
Epoch 7/50 | train loss 0.5240 valid% 37.8 | val loss 0.7784 valid% 18.1
Epoch 8/50 | train loss 0.5378 valid% 36.6 | val loss 0.6223 valid% 35.9
Epoch 9/50 | train loss 0.4857 valid% 42.0 | val loss 0.6863 valid% 27.8
Epoch 10/50 | train loss 0.4380 valid% 47.8 | val loss 1.2443 valid% 27.3
Epoch 11/50 | train loss 0.9787 valid% 13.7 | val loss 0.7629 valid% 17.8
Epoch 12/50 | train loss 0.6420 valid% 26.1 | val loss 0.6698 valid% 28.7
Epoch 13/50 | train loss 0.5230 valid% 37.6 | val loss 0.5403 valid% 37.6
Epoch 14/50

In [ ]:
num_epochs = 50
bs = 2048
sigma = 0.5
denoiser_lr = 1e-2
audio_decoder_lr = 5e-2
audio_dim = 10

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# perm = torch.randperm(all_sequences.size(0))
# all_sequences = all_sequences[perm]

# train_size = int(0.8 * all_sequences.size(0))
# train_data = all_sequences[:train_size]
# val_data = all_sequences[train_size:]

# train_loader = make_epoch_loader(train_data, bs, shuffle=True)
# val_loader = make_epoch_loader(val_data, bs, shuffle=False)

model = Denoiser(vocab_size=vocab_size, seq_len=SEQ_LENGTH, audio_dim=audio_dim).to(device)
mu_nn = AudioDecoder(audio_dim=audio_dim, vocab_size=vocab_size).to(device)
param_groups = [
    {"params": model.parameters(), "lr": denoiser_lr},
    {"params": mu_nn.parameters(), "lr": audio_decoder_lr},
]
optimizer = torch.optim.AdamW(param_groups)


interpolant_fn = partial(
    make_interpolant,
    alpha_fn=lambda t: 1 - t,
    beta_fn=lambda t: t,
    gamma_fn=lambda t: 1 - t,
)

steps_per_epoch = len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=max_lr,
    total_steps=steps_per_epoch * num_epochs,
)

history = {"train": [], "val": []}

# confusion_noise = 0.3
# Phi, confusable_pairs = make_phi(
#     vocab_size, audio_dim, num_confusable_pairs=2, confusion_noise=confusion_noise, device=device
# )
# print(confusable_pairs)

# eta = 1
# kernel = torch.tensor([0.1, 0.8, 0.1], device=device).view(1, 1, 3)
# audio_proxy_fn = partial(
#     make_audio_proxy, Phi=Phi, kernel=kernel, eta=eta, audio_dim=audio_dim
# )

for epoch in range(num_epochs):
    train_metrics = run_epoch(
        model,
        train_loader,
        transition_matrix,
        device,
        optimizer=optimizer,
        scheduler=scheduler,
        sigma=sigma,
        audio_proxy_fn=audio_proxy_fn,
        require_even_sum=require_even_sum,
        interpolant_fn=interpolant_fn,
        mu_nn = mu_nn,
    )
    val_metrics = run_epoch(
        model,
        val_loader,
        transition_matrix,
        device,
        optimizer=None,
        scheduler=None,
        sigma=sigma,
        audio_proxy_fn=audio_proxy_fn,
        require_even_sum=require_even_sum,
        interpolant_fn=interpolant_fn,
        mu_nn = mu_nn,
    )

    history["train"].append(train_metrics)
    history["val"].append(val_metrics)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"train loss {train_metrics['loss']:.4f} valid% {train_metrics['frac_valid']*100:.1f} | "
        f"val loss {val_metrics['loss']:.4f} valid% {val_metrics['frac_valid']*100:.1f}"
    )

Epoch 1/50 | train loss 3.8049 valid% 7.4 | val loss 2.9310 valid% 12.7
Epoch 2/50 | train loss 2.5291 valid% 18.5 | val loss 2.1510 valid% 24.0
Epoch 3/50 | train loss 1.8684 valid% 29.5 | val loss 1.6397 valid% 34.0
Epoch 4/50 | train loss 1.5401 valid% 34.8 | val loss 1.4770 valid% 35.9
Epoch 5/50 | train loss 1.4501 valid% 36.8 | val loss 1.4432 valid% 36.0
Epoch 6/50 | train loss 1.4206 valid% 37.5 | val loss 1.4740 valid% 32.6
Epoch 7/50 | train loss 1.4201 valid% 36.8 | val loss 1.5212 valid% 32.0
Epoch 8/50 | train loss 1.4123 valid% 37.3 | val loss 1.6242 valid% 25.8
Epoch 9/50 | train loss 1.4148 valid% 37.5 | val loss 1.6531 valid% 32.1
Epoch 10/50 | train loss 1.4706 valid% 33.0 | val loss 1.5157 valid% 30.8
Epoch 11/50 | train loss 1.3985 valid% 38.8 | val loss 1.4298 valid% 39.0
Epoch 12/50 | train loss 1.3010 valid% 49.0 | val loss 1.6203 valid% 33.3
Epoch 13/50 | train loss 1.4189 valid% 40.0 | val loss 1.9360 valid% 28.0
Epoch 14/50 | train loss 1.3862 valid% 42.8 | va

In [ ]:
num_epochs = 50
bs = 2048
sigma = 0.5
denoiser_lr = 1e-2
audio_decoder_lr = 5e-2
audio_dim = 10

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# perm = torch.randperm(all_sequences.size(0))
# all_sequences = all_sequences[perm]

# train_size = int(0.8 * all_sequences.size(0))
# train_data = all_sequences[:train_size]
# val_data = all_sequences[train_size:]

# train_loader = make_epoch_loader(train_data, bs, shuffle=True)
# val_loader = make_epoch_loader(val_data, bs, shuffle=False)

model = Denoiser(vocab_size=vocab_size, seq_len=SEQ_LENGTH, audio_dim=audio_dim).to(device)
mu_nn = AudioDecoder(audio_dim=audio_dim, vocab_size=vocab_size).to(device)
param_groups = [
    {"params": model.parameters(), "lr": denoiser_lr},
    {"params": mu_nn.parameters(), "lr": audio_decoder_lr},
]
optimizer = torch.optim.AdamW(param_groups)


interpolant_fn = partial(
    make_interpolant,
    alpha_fn=lambda t: 1 - t,
    beta_fn=lambda t: t,
    gamma_fn=lambda t: 2 * t * (1 - t),
)

steps_per_epoch = len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=max_lr,
    total_steps=steps_per_epoch * num_epochs,
)

history = {"train": [], "val": []}

# confusion_noise = 0.3
# Phi, confusable_pairs = make_phi(
#     vocab_size, audio_dim, num_confusable_pairs=2, confusion_noise=confusion_noise, device=device
# )
# print(confusable_pairs)

# eta = 1
# kernel = torch.tensor([0.1, 0.8, 0.1], device=device).view(1, 1, 3)
# audio_proxy_fn = partial(
#     make_audio_proxy, Phi=Phi, kernel=kernel, eta=eta, audio_dim=audio_dim
# )

for epoch in range(num_epochs):
    train_metrics = run_epoch(
        model,
        train_loader,
        transition_matrix,
        device,
        optimizer=optimizer,
        scheduler=scheduler,
        sigma=sigma,
        audio_proxy_fn=audio_proxy_fn,
        require_even_sum=require_even_sum,
        interpolant_fn=interpolant_fn,
        mu_nn = mu_nn,
    )
    val_metrics = run_epoch(
        model,
        val_loader,
        transition_matrix,
        device,
        optimizer=None,
        scheduler=None,
        sigma=sigma,
        audio_proxy_fn=audio_proxy_fn,
        require_even_sum=require_even_sum,
        interpolant_fn=interpolant_fn,
        mu_nn = mu_nn,
    )

    history["train"].append(train_metrics)
    history["val"].append(val_metrics)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"train loss {train_metrics['loss']:.4f} valid% {train_metrics['frac_valid']*100:.1f} | "
        f"val loss {val_metrics['loss']:.4f} valid% {val_metrics['frac_valid']*100:.1f}"
    )

Epoch 1/50 | train loss 3.7506 valid% 7.5 | val loss 2.9455 valid% 13.5
Epoch 2/50 | train loss 2.5613 valid% 16.6 | val loss 2.1859 valid% 19.4
Epoch 3/50 | train loss 1.9411 valid% 21.0 | val loss 1.7433 valid% 21.9
Epoch 4/50 | train loss 1.5980 valid% 28.9 | val loss 1.5186 valid% 30.0
Epoch 5/50 | train loss 1.4664 valid% 34.4 | val loss 1.5673 valid% 28.4
Epoch 6/50 | train loss 1.4557 valid% 34.2 | val loss 1.5207 valid% 27.5
Epoch 7/50 | train loss 1.4351 valid% 35.0 | val loss 1.6648 valid% 21.0
Epoch 8/50 | train loss 1.4333 valid% 34.9 | val loss 1.7870 valid% 19.0
Epoch 9/50 | train loss 1.4938 valid% 30.7 | val loss 1.5563 valid% 25.7
Epoch 10/50 | train loss 1.3966 valid% 38.0 | val loss 1.7814 valid% 19.2
Epoch 11/50 | train loss 1.3864 valid% 40.7 | val loss 1.6810 valid% 32.5
Epoch 12/50 | train loss 1.3714 valid% 44.3 | val loss 1.5560 valid% 37.8
Epoch 13/50 | train loss 1.2432 valid% 57.9 | val loss 1.4521 valid% 44.0
Epoch 14/50 | train loss 1.2162 valid% 61.0 | va